## Deploy a Model to Microsoft Foundry with Python

In this notebook, you will learn how to **deploy an AI model** to your Foundry project — entirely with Python code. Instead of clicking through the portal, we will:

1. Connect to Azure and list available models
2. Pick a model and deploy it programmatically
3. List all active deployments in our project
4. Test the deployed model by chatting with it

By the end, you will have a fully functional model endpoint that you can use from other notebooks in this folder.

### Step 1: Install the Python Packages We Need

We need two types of SDKs for this demo:
- **`azure-mgmt-cognitiveservices`** — the Azure management SDK that lets us create model deployments on our Cognitive Services resource
- **`azure-ai-projects`** — the Foundry SDK that lets us list deployments and test them

We also install `azure-identity` (for secure login), `python-dotenv` (to load settings from a file), and `openai` (to test the deployed model).

In [4]:
# Install all the packages we need:
# - azure-mgmt-cognitiveservices: manage deployments on Azure Cognitive Services
# - azure-ai-projects: list deployments and get an OpenAI client for testing
# - openai: the OpenAI library used behind the scenes for chat
# - python-dotenv: reads our .env configuration file
# - azure-identity: handles Azure login securely
%pip install azure-mgmt-cognitiveservices>=13.5.0,<15.0.0 azure-ai-projects==2.2.0 openai==2.9.0 python-dotenv==1.0.1 azure-identity

Note: you may need to restart the kernel to use updated packages.


The system cannot find the file specified.


### Step 2: Load Our Configuration

We store sensitive configuration in a `.env` file. For this notebook we need a few extra values because the management SDK needs to know *which* Azure resource to talk to.

Your `.env` file should contain:
- `FOUNDRY_PROJECT_ENDPOINT` — the Foundry project URL (used later to test the model)
- `AZURE_SUBSCRIPTION_ID` — your Azure subscription ID
- `AZURE_RESOURCE_GROUP` — the resource group that contains your Foundry resource
- `FOUNDRY_ACCOUNT_NAME` — the name of your Azure AI Services / Cognitive Services account (the parent resource, not the project)

In [ ]:
# 'os' lets us access environment variables
import os

# 'load_dotenv' reads key-value pairs from a .env file into environment variables
from dotenv import load_dotenv

# Load the .env file — this must run before we read any values
load_dotenv()

# Read each setting from the environment
my_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")       # Foundry project URL (for testing later)
subscription_id = os.getenv("AZURE_SUBSCRIPTION_ID")       # Azure subscription ID
resource_group = os.getenv("AZURE_RESOURCE_GROUP")         # Resource group name
account_name = os.getenv("FOUNDRY_ACCOUNT_NAME")           # Cognitive Services account name

# Quick check — make sure nothing is missing
print("Endpoint:      ", my_endpoint)
print("Subscription:  ", subscription_id)
print("Resource group:", resource_group)
print("Account name:  ", account_name)

if not my_endpoint or not subscription_id or not resource_group or not account_name:
    print("WARNING: One or more values are empty. Fill in your .env file before continuing.")

Endpoint:       None
Subscription:   None
Resource group: None
Account name:   None


### Step 3: Connect to Azure

We need **two clients** in this notebook, each serving a different purpose:

| Client | SDK | What it does |
|---|---|---|
| `CognitiveServicesManagementClient` | `azure-mgmt-cognitiveservices` | List available models, create/delete deployments |
| `AIProjectClient` | `azure-ai-projects` | List existing deployments, test models via OpenAI client |

Think of the management client as the construction crew that *builds* the deployment, and `AIProjectClient` as the front desk that *uses* it.

In [3]:
import sys

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

Python executable:
c:\Users\Ashok\AppData\Local\Programs\Python\Python312\python.exe

Python version:
3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]


In [4]:
%pip install "azure-mgmt-cognitiveservices>=13.5.0,<15.0.0"

  Using cached azure_mgmt_cognitiveservices-14.1.0-py3-none-any.whl.metadata (32 kB)
  Using cached msrest-0.7.1-py3-none-any.whl.metadata (21 kB)
  Using cached azure_mgmt_core-1.6.0-py3-none-any.whl.metadata (4.6 kB)
  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl.metadata (11 kB)
  Using cached oauthlib-4.0.0-py3-none-any.whl.metadata (8.6 kB)
Using cached azure_mgmt_cognitiveservices-14.1.0-py3-none-any.whl (290 kB)
Using cached azure_mgmt_core-1.6.0-py3-none-any.whl (29 kB)
Using cached msrest-0.7.1-py3-none-any.whl (85 kB)
Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl (24 kB)
Using cached oauthlib-4.0.0-py3-none-any.whl (159 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
%pip install "azure-ai-projects==2.2.0" "openai==2.9.0" "python-dotenv==1.0.1" azure-identity

  Using cached python_dotenv-1.0.1-py3-none-any.whl.metadata (23 kB)
Using cached python_dotenv-1.0.1-py3-none-any.whl (19 kB)
  Attempting uninstall: python-dotenv
    Found existing installation: python-dotenv 1.2.4
    Uninstalling python-dotenv-1.2.4:
      Successfully uninstalled python-dotenv-1.2.4
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from azure.identity import DefaultAzureCredential
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from azure.ai.projects import AIProjectClient

print("All Azure imports successful!")

All Azure imports successful!


In [14]:
import os
from dotenv import load_dotenv

load_dotenv()

subscription_id = os.getenv("AZURE_SUBSCRIPTION_ID")

print("Subscription ID loaded:", subscription_id is not None)

Subscription ID loaded: True


In [15]:
import os

print("Current folder:")
print(os.getcwd())

print("\n.env exists:")
print(os.path.exists(".env"))

Current folder:
c:\Users\Ashok\OneDrive\Documents\Microsoft-Foundry-Source-Code\Microsoft-Foundry-Source-Code\Foundry-Models\Deploy-Model

.env exists:
True


In [19]:
from pathlib import Path
from dotenv import load_dotenv

env_path = Path.cwd() / ".env"

print("Looking for:")
print(env_path)

print("Exists:", env_path.exists())

load_dotenv(env_path)

print("Subscription ID loaded:",
      bool(os.getenv("AZURE_SUBSCRIPTION_ID")))

Looking for:
c:\Users\Ashok\OneDrive\Documents\Microsoft-Foundry-Source-Code\Microsoft-Foundry-Source-Code\Foundry-Models\Deploy-Model\.env
Exists: True
Subscription ID loaded: True


In [21]:
from azure.identity import DefaultAzureCredential
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from azure.ai.projects import AIProjectClient

credential = DefaultAzureCredential()

mgmt_client = CognitiveServicesManagementClient(
    credential=credential,
    subscription_id="77bbf218-df70-4269-955d-4818a0834a6",
)

print("Management client connected!")

foundry_client = AIProjectClient(
    endpoint="https://foundrydemodemo369.services.ai.azure.com/api/projects/foundry-demo",
    credential=credential,
)

print("AIProjectClient connected!")

Management client connected!
AIProjectClient connected!


In [10]:
# DefaultAzureCredential handles Azure login automatically
# It tries multiple login methods (VS Code, Azure CLI, etc.) until one works
from azure.identity import DefaultAzureCredential

# CognitiveServicesManagementClient lets us manage deployments on our Azure AI resource
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient

# AIProjectClient is the Foundry SDK client — we use it to list and test deployments
from azure.ai.projects import AIProjectClient

# Create the credential object (shared by both clients)
credential = DefaultAzureCredential()

# --- Client 1: Management client (for deploying models) ---
# This client talks to the Azure Resource Manager (management plane)
mgmt_client = CognitiveServicesManagementClient(
    credential=credential,
    subscription_id="77bbf218-df70-4269-955d-4818a0834a6",
)
print("Management client connected!")

# --- Client 2: AIProjectClient (for listing deployments and testing) ---
# This client talks to the Foundry project (data plane)
foundry_client = AIProjectClient(
    endpoint="https://foundrydemodemo369.services.ai.azure.com/api/projects/foundry-demo",
    credential=credential,
)
print("AIProjectClient connected!")

Management client connected!
AIProjectClient connected!


In [1]:
import sys

print(sys.executable)
print(sys.version)

c:\Users\Ashok\OneDrive\Documents\Microsoft-Foundry-Source-Code\Microsoft-Foundry-Source-Code\.venv\Scripts\python.exe
3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]


In [3]:
%pip install "azure-mgmt-cognitiveservices>=13.5.0,<15.0.0" "azure-ai-projects==2.2.0" "openai==2.9.0" "python-dotenv==1.0.1" azure-identity

  Using cached azure_mgmt_cognitiveservices-14.1.0-py3-none-any.whl.metadata (32 kB)
  Using cached azure_ai_projects-2.2.0-py3-none-any.whl.metadata (56 kB)
  Using cached openai-2.9.0-py3-none-any.whl.metadata (29 kB)
  Using cached python_dotenv-1.0.1-py3-none-any.whl.metadata (23 kB)
  Using cached azure_identity-1.26.0-py3-none-any.whl.metadata (93 kB)
  Using cached isodate-0.7.2-py3-none-any.whl.metadata (11 kB)
  Using cached azure_core-1.41.0-py3-none-any.whl.metadata (49 kB)
  Using cached azure_storage_blob-12.31.0-py3-none-any.whl.metadata (26 kB)
  Using cached anyio-4.15.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jiter-0.17.0-cp312-cp312-win_amd64.whl.metadata (5.3 kB)
  Using cached pydantic-2.13.5-py3-none-any.whl.metadata (110 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl

In [1]:
import sys

print(sys.executable)

c:\Users\Ashok\OneDrive\Documents\Microsoft-Foundry-Source-Code\Microsoft-Foundry-Source-Code\.venv\Scripts\python.exe


In [2]:
from azure.identity import DefaultAzureCredential
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from azure.ai.projects import AIProjectClient

print("Azure packages OK!")

Azure packages OK!


In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

subscription_id = os.getenv("AZURE_SUBSCRIPTION_ID")
resource_group = os.getenv("AZURE_RESOURCE_GROUP")
account_name = os.getenv("FOUNDRY_ACCOUNT_NAME")
my_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")

print("Subscription ID:", bool(subscription_id))
print("Resource Group:", bool(resource_group))
print("Account Name:", bool(account_name))
print("Foundry Endpoint:", bool(my_endpoint))

Subscription ID: True
Resource Group: True
Account Name: True
Foundry Endpoint: True


In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

subscription_id = os.getenv("AZURE_SUBSCRIPTION_ID")
resource_group = os.getenv("AZURE_RESOURCE_GROUP")
account_name = os.getenv("FOUNDRY_ACCOUNT_NAME")
my_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")

print("Subscription ID:", bool(subscription_id))
print("Resource Group:", bool(resource_group))
print("Account Name:", bool(account_name))
print("Foundry Endpoint:", bool(my_endpoint))

Subscription ID: True
Resource Group: True
Account Name: True
Foundry Endpoint: True


In [6]:
from azure.identity import DefaultAzureCredential
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from azure.ai.projects import AIProjectClient

credential = DefaultAzureCredential()

mgmt_client = CognitiveServicesManagementClient(
    credential=credential,
    subscription_id=subscription_id,
)

print("Management client connected!")

foundry_client = AIProjectClient(
    endpoint=my_endpoint,
    credential=credential,
)

print("AIProjectClient connected!")

Management client connected!
AIProjectClient connected!


In [11]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

subscription_id = os.getenv("AZURE_SUBSCRIPTION_ID")
resource_group = os.getenv("AZURE_RESOURCE_GROUP")
account_name = os.getenv("FOUNDRY_ACCOUNT_NAME")
my_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")

print("Subscription loaded:", bool(subscription_id))
print("Resource group:", resource_group)
print("Account:", account_name)
print("Endpoint loaded:", bool(my_endpoint))

Subscription loaded: True
Resource group: foundry
Account: foundrydemodemo369
Endpoint loaded: True


In [12]:
available_models = mgmt_client.accounts.list_models(
    resource_group_name=resource_group,
    account_name=account_name,
)

print("API call created successfully")
print(type(available_models))

API call created successfully
<class 'azure.core.paging.ItemPaged'>


In [14]:
from azure.identity import DefaultAzureCredential
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient

credential = DefaultAzureCredential()

mgmt_client = CognitiveServicesManagementClient(
    credential=credential,
    subscription_id=subscription_id,
)

print("Management client connected!")

Management client connected!


In [15]:
available_models = mgmt_client.accounts.list_models(
    resource_group_name=resource_group,
    account_name=account_name,
)

models = list(available_models)

print("Number of models:", len(models))

Number of models: 136


### Step 4: Browse Available Models

Before deploying a model, let's see which ones are available in our Azure region. Not every model is available everywhere — this step shows you exactly what you can deploy.

We will filter for **OpenAI** format models (GPT, DALL-E, Whisper, embeddings, etc.) since those are the most commonly used.

In [16]:
# List all models available for our Cognitive Services account
# This returns models that can be deployed in the region where our account lives
available_models = mgmt_client.accounts.list_models(
    resource_group_name=resource_group,
    account_name=account_name,
)

# Collect and display the models, grouped by format
# 'format' is usually 'OpenAI' for GPT models or other publisher names
print("Available models for deployment:\n")
model_count = 0
for model in available_models:
    model_count += 1
    # Show model name, version, and what it can do
    capabilities = ", ".join(model.capabilities.keys()) if model.capabilities else "N/A"
    print(f"  {model_count:3d}. {model.name:35s}  v{model.version:15s}  [{capabilities}]")
    # Stop after 30 to keep the output manageable
    if model_count >= 30:
        print(f"\n  ... showing first 30 of many models.")
        break

print(f"\nTip: Visit https://ai.azure.com/catalog to browse the full catalog with descriptions.")

Available models for deployment:

    1. whisper                              v001              [scaleType, audio, audioTranscriptions, audioTranslations, area]
    2. gpt-4o                               v2024-05-13       [area, chatCompletion, jsonObjectResponse, maxContextToken, maxOutputToken, assistants, FineTuneTokensMaxValuePerExample, FineTuneTokensMaxValue, allowProvisionedManagedCommitment, responses, agentsV2]
    3. gpt-4o                               v2024-08-06       [area, chatCompletion, globalFineTune, datazoneFineTune, devTierFineTune, assistants, jsonSchemaResponse, maxContextToken, maxOutputToken, FineTuneTokensMaxValuePerExample, FineTuneTokensMaxValue, allowProvisionedManagedCommitment, responses, agentsV2]
    4. gpt-4o-mini                          v2024-07-18       [area, chatCompletion, jsonObjectResponse, maxContextToken, maxOutputToken, assistants, globalFineTune, datazoneFineTune, devTierFineTune, FineTuneTokensMaxValuePerExample, FineTuneTokensMaxValue, a

### Step 5: Choose a Model to Deploy

Let's deploy **GPT-5.1** — a generally available general-purpose model from OpenAI and the recommended replacement for the GPT-4o generation. (Newer GPT-5.x models exist; GPT-5.1 has one of the longest support windows, to 2027-05-15.)

> **Why GPT-5.1?** It is generally available and it is the default model used throughout the labs, so every other lab that expects a `gpt-5.1` deployment works against the same endpoint. Swap in a different model by changing the `model_name` and `model_version` below.

> **Why not a GPT-4 generation model?** `gpt-4o`, `gpt-4o-mini`, `gpt-4.1`, and `gpt-4.1-mini` are all **Deprecated**. Deprecated means a subscription that has never deployed that exact model version can no longer create a deployment of it — so on a new subscription those models are unavailable, even though they still serve traffic for existing customers until their retirement date. Always check the model retirement schedule before choosing a model.

> **Note:** The `model_version` below was current at the time of recording. If the deployment fails because the version has been retired, check the output from Step 4 for the latest available version and update it accordingly.

First, let's look up the exact details of this model to confirm it is available.

In [17]:
# The model we want to deploy
# Change these values if you want to deploy a different model
model_name = "gpt-5.1"                 # The model name as it appears in the catalog
model_format = "OpenAI"                # The model format (OpenAI for GPT models)
# Update this version if the model listing (Step 4) shows a newer version available
model_version = "2025-11-13"           # The model version (check Step 4 output for available versions)

# The name we will give our deployment — this is how we reference it later
# Must be unique within our account and can contain letters, numbers, and hyphens
deployment_name = "gpt-5.1"

# Verify the model exists and find its supported SKU
# Different models support different SKUs (e.g., 'Standard', 'GlobalStandard', etc.)
# We will automatically pick the first available SKU so the deployment works correctly
found = False
model_sku_name = None
model_sku_capacity = None

for model in mgmt_client.accounts.list_models(
    resource_group_name=resource_group,
    account_name=account_name,
):
    if model.name == model_name and model.version == model_version:
        found = True
        capabilities = ", ".join(model.capabilities.keys()) if model.capabilities else "N/A"
        print(f"Model found!")
        print(f"  Name:         {model.name}")
        print(f"  Version:      {model.version}")
        print(f"  Format:       {model.format}")
        print(f"  Capabilities: {capabilities}")

        # Show all available SKUs and pick the first one for deployment
        if model.skus:
            sku_names = [s.name for s in model.skus]
            print(f"  Available SKUs: {', '.join(sku_names)}")

            # Use the first SKU — this is typically the recommended one
            first_sku = model.skus[0]
            model_sku_name = first_sku.name
            model_sku_capacity = first_sku.capacity.default if first_sku.capacity else 10
            print(f"\n  Selected SKU for deployment:")
            print(f"    SKU:      {model_sku_name}")
            print(f"    Capacity: {model_sku_capacity}")
        break

if not found:
    print(f"Model '{model_name}' version '{model_version}' was not found!")
    print("Check the output from Step 4 for available models and versions.")

Model found!
  Name:         gpt-5.1
  Version:      2025-11-13
  Format:       OpenAI
  Capabilities: area, chatCompletion, responses, assistants, priorityTierSkus
  Available SKUs: Standard, GlobalStandard, DataZoneStandard, ProvisionedManaged, DataZoneProvisionedManaged, GlobalProvisionedManaged, GlobalBatch, DataZoneBatch

  Selected SKU for deployment:
    SKU:      Standard
    Capacity: 10


### Step 6: Deploy the Model

This is the main event! We will create a deployment using the SKU that was automatically selected in Step 5. Different models support different SKUs — for example, `gpt-5.1` offers **Standard** (regional) and **GlobalStandard** (globally load-balanced) among others, and the cell above selects whichever the catalog lists first.

We need to specify:
- **Deployment name** — how we will reference it later
- **Model details** — name, version, and format
- **SKU** — the pricing tier (auto-detected from Step 5)
- **Capacity** — the rate limit in thousands of tokens per minute (TPM)

In [18]:
# Import the classes we need to define the deployment
from azure.mgmt.cognitiveservices.models import (
    Deployment,              # The top-level deployment object
    DeploymentProperties,    # Configuration properties (model, version upgrade policy, etc.)
    DeploymentModel,         # Specifies which model to deploy
    Sku,                     # The pricing tier and capacity
)

if not found or model_sku_name is None:
    print("ERROR: Model not found in Step 5. Cannot deploy. Check the model name and version above.")
else:
    # Build the deployment definition
    # This is like filling out a form that tells Azure exactly what to create
    deployment = Deployment(
        properties=DeploymentProperties(
            model=DeploymentModel(
                name=model_name,             # Which model (e.g., 'gpt-5.1')
                version=model_version,       # Which version (e.g., '2025-11-13')
                format=model_format,         # Which format (e.g., 'OpenAI')
            ),
            # version_upgrade_option controls automatic updates:
            #   'NoAutoUpgrade' = stay on this exact version until we change it manually
            #   'OnceNewDefaultVersionAvailable' = auto-upgrade when a new default is released
            version_upgrade_option="NoAutoUpgrade",
        ),
        sku=Sku(
            name=model_sku_name,         # SKU auto-detected in Step 5 (e.g., 'GlobalStandard')
            capacity=model_sku_capacity,  # Default capacity from the model's SKU definition
        ),
    )

    print(f"Deployment configuration:")
    print(f"  Model:    {model_name} v{model_version}")
    print(f"  SKU:      {model_sku_name}")
    print(f"  Capacity: {model_sku_capacity}")

    # Submit the deployment request and wait for it to finish
    # begin_create_or_update() starts the process; .result() waits until it is done
    print(f"\nDeploying '{model_name}' as '{deployment_name}'...")
    print("This usually takes about 1 minute. Please wait...\n")

    result = mgmt_client.deployments.begin_create_or_update(
        resource_group_name=resource_group,
        account_name=account_name,
        deployment_name=deployment_name,
        deployment=deployment,
    ).result()

    # Print the result
    print(f"Deployment complete!")
    print(f"  Name:   {result.name}")
    print(f"  Model:  {result.properties.model.name} v{result.properties.model.version}")
    print(f"  Status: {result.properties.provisioning_state}")
    print(f"  SKU:    {result.sku.name} (capacity: {result.sku.capacity})")

Deployment configuration:
  Model:    gpt-5.1 v2025-11-13
  SKU:      Standard
  Capacity: 10

Deploying 'gpt-5.1' as 'gpt-5.1'...
This usually takes about 1 minute. Please wait...

Deployment complete!
  Name:   gpt-5.1
  Model:  gpt-5.1 v2025-11-13
  Status: Succeeded
  SKU:    Standard (capacity: 10)


### Step 7: List All Deployments in Your Project

Now let's use the `AIProjectClient` (the Foundry SDK) to see all the models currently deployed in your project — including the one we just created.

In [19]:
# List all model deployments in the Foundry project
# This shows everything — models deployed from the portal AND from code
print("All model deployments in your Foundry project:\n")

for dep in foundry_client.deployments.list():
    print(f"  Name:      {dep.name}")
    print(f"  Model:     {dep.model_name}")
    print(f"  Publisher: {dep.model_publisher}")
    print(f"  Version:   {dep.model_version}")
    print(f"  Type:      {dep.type}")
    print()

All model deployments in your Foundry project:

  Name:      gpt-4.1
  Model:     gpt-4.1
  Publisher: OpenAI
  Version:   2025-04-14
  Type:      ModelDeployment

  Name:      text-embedding-3-small
  Model:     text-embedding-3-small
  Publisher: OpenAI
  Version:   1
  Type:      ModelDeployment

  Name:      gpt-5.1
  Model:     gpt-5.1
  Publisher: OpenAI
  Version:   2025-11-13
  Type:      ModelDeployment



### Step 8: Test the Deployed Model

The moment of truth! Let's send a question to the model we just deployed and see if it responds. Using the Responses API pattern — we get an OpenAI-compatible client from the Foundry project client, then call `responses.create()` with our deployment name.

In [21]:
# Get an OpenAI-compatible client from our Foundry client
# This handles authentication automatically — no API keys needed
chat_client = foundry_client.get_openai_client()

# Send a test question to our newly deployed model
# We use the deployment_name as the 'model' parameter (not the catalog model name)
print(f"Sending a test question to '{deployment_name}'...\n")

ai_response = chat_client.responses.create(
    model=deployment_name,               # Use the deployment name we chose in Step 5
    instructions="You are a helpful and concise assistant.",
    input="Explain what a model deployment is in microsoft Foundry in two sentences."
)

# Print the model's response
print(f"The model replied:\n{ai_response.output_text}")

Sending a test question to 'gpt-5.1'...

The model replied:
In Microsoft Fabric (Foundry), a model deployment is the process of taking a trained machine learning model and turning it into a usable, managed endpoint (such as a REST API) that other applications or services can call. It handles packaging the model, provisioning the necessary compute, and managing versions, scaling, and monitoring so the model can be reliably used in production.
